<a href="https://colab.research.google.com/github/SaluLink-Design/Authi-experiment-PMB/blob/main/Authi_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 97.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0


In [3]:
# Load model directly
from transformers import AutoModel
model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT", dtype="auto")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
!git clone https://github.com/SaluLink-Design/experiment.git

Cloning into 'experiment'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 9.74 KiB | 1.22 MiB/s, done.


In [5]:
import pandas as pd

# Load the PMB conditions dataset
csv_file_path = '/content/experiment/prescribed-minimum-benefit-chronic-medicine-list-formulary 2.csv'
pmb_df = pd.read_csv(csv_file_path)

# Display the first few rows and column information to understand the data
display(pmb_df.head())
print(pmb_df.info())

,PMB CODE,PMB CONDITION,MEDICINE SUB-CATEGORY,MEDICINE CLASS NAME,PRODUCT NAME,STRENGTH
0,100E,Patent ductus arteriosus; aortic pulmonary fis...,"Ace Inhibitors, Plain",Captopril,Mylan captopril,25 mg
1,NaN,NaN,NaN,NaN,Mylan captopril,50 mg
2,NaN,NaN,NaN,Enalapril,Tenezil,10 mg
3,NaN,NaN,NaN,NaN,Natrapress,10 mg
4,NaN,NaN,NaN,NaN,Enalapril 10 arya,10 mg


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1497 entries, 0 to 1496
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   PMB CODE               109 non-null    object
 1   PMB CONDITION          112 non-null    object
 2   MEDICINE SUB-CATEGORY  241 non-null    object
 3   MEDICINE CLASS NAME    360 non-null    object
 4   PRODUCT NAME           1497 non-null   object
 5   STRENGTH               1497 non-null   object
dtypes: object(6)
memory usage: 70.3+ KB
None


In [6]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load the tokenizer for Bio_ClinicalBERT
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")

# Assuming 'model' was already loaded in a previous cell
# model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT", dtype="auto")

def get_clinical_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Get the embeddings for the [CLS] token (sentence embedding)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()

# Example usage:
clinical_note = "Patient presents with weakness, loss of reflexes, acute onset, and suspected stroke."
embeddings = get_clinical_embeddings(clinical_note)
print("Clinical Note Embeddings Shape:", embeddings.shape)

Clinical Note Embeddings Shape: (768,)


In [7]:
from tqdm.auto import tqdm

tqdm.pandas() # Enable tqdm for pandas apply

# Fill NaN values in 'PMB CONDITION' with an empty string before generating embeddings
pmb_df['PMB CONDITION_filled'] = pmb_df['PMB CONDITION'].fillna('')

# Generate embeddings for the 'PMB CONDITION' column
pmb_df['PMB_CONDITION_EMBEDDING'] = pmb_df['PMB CONDITION_filled'].progress_apply(get_clinical_embeddings)

print("Embeddings for PMB conditions generated successfully.")
# Display the first few rows with the new embedding column
display(pmb_df[['PMB CONDITION', 'PMB_CONDITION_EMBEDDING']].head())

  0%|          | 0/1497 [00:00<?, ?it/s]

Embeddings for PMB conditions generated successfully.


,PMB CONDITION,PMB_CONDITION_EMBEDDING
0,Patent ductus arteriosus; aortic pulmonary fis...,"[0.17243156, 0.05518842, 0.001594775, 0.014159..."
1,NaN,"[0.22482204, -0.24363495, -0.03791469, 0.12638..."
2,NaN,"[0.22482204, -0.24363495, -0.03791469, 0.12638..."
3,NaN,"[0.22482204, -0.24363495, -0.03791469, 0.12638..."
4,NaN,"[0.22482204, -0.24363495, -0.03791469, 0.12638..."


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def find_most_similar_pmb_conditions(clinical_note_text, pmb_df, top_n=5):
    # 1. Get embedding for the clinical note
    note_embedding = get_clinical_embeddings(clinical_note_text)

    # Convert the note_embedding to a 2D array for cosine_similarity (expected format)
    note_embedding_2d = note_embedding.reshape(1, -1)

    # Ensure PMB condition embeddings are also in a 2D array and handle potential empty/None embeddings
    # Filter out rows where 'PMB_CONDITION_EMBEDDING' might be None or a non-array type
    valid_embeddings_df = pmb_df[pmb_df['PMB_CONDITION_EMBEDDING'].apply(lambda x: isinstance(x, np.ndarray) and x.size > 0)]

    if valid_embeddings_df.empty:
        print("No valid PMB condition embeddings found for comparison.")
        return pd.DataFrame()

    pmb_embeddings_2d = np.array(valid_embeddings_df['PMB_CONDITION_EMBEDDING'].tolist())

    # 2. Calculate cosine similarity between the clinical note and all PMB conditions
    similarities = cosine_similarity(note_embedding_2d, pmb_embeddings_2d).flatten()

    # Add similarities to the valid_embeddings_df
    valid_embeddings_df = valid_embeddings_df.copy()
    valid_embeddings_df['similarity'] = similarities

    # 3. Sort by similarity and get the top N
    top_matches = valid_embeddings_df.sort_values(by='similarity', ascending=False).head(top_n)

    return top_matches[['PMB CONDITION', 'MEDICINE SUB-CATEGORY', 'MEDICINE CLASS NAME', 'PRODUCT NAME', 'STRENGTH', 'similarity']]

# Example usage with the previously defined clinical_note
print("\nFinding top 5 PMB conditions for the clinical note:")
print(f'"{clinical_note}"')
top_pmb_matches = find_most_similar_pmb_conditions(clinical_note, pmb_df, top_n=5)
display(top_pmb_matches)


Finding top 5 PMB conditions for the clinical note:
"Patient presents with weakness, loss of reflexes, acute onset, and suspected stroke."


,PMB CONDITION,MEDICINE SUB-CATEGORY,MEDICINE CLASS NAME,PRODUCT NAME,STRENGTH,similarity
1244,"Acute generalized paralysis, including polio a...",Tricyclic antidepressants,Amitriptyline,Amitriptyline 10 mg austell,10 mg,0.880483
755,"Hyperplasia of the prostate, with acute urinar...","Antibacterial Agents, systemic",Sulfamethoxazole and trimethoprim,Cozole,480 mg,0.846068
692,"Hyperplasia of the prostate, with acute urinar...","Antibacterial Agents, systemic",Amoxicillin,Moxymax s,125 mg/5 mL,0.846068
723,"Hyperplasia of the prostate, with acute urinar...","Antibacterial Agents, systemic",Amoxicillin and enzyme inhibitor,Auro-amoxiclav,625 mg,0.846068
680,"Hyperplasia of the prostate, with acute urinar...",Antiadrenergic agents - Alpha-adrenoreceptor a...,Doxazosin,Cardugen (was merck-doxazosin),1 mg,0.846068
